### i220524 FAST-NUCES Assignment 2 — Responsible & Explainable AI


## Environment setup

Install dependencies (local or Colab), then continue. On Colab: **Runtime → Change runtime type → GPU**.



In [14]:
"""Install dependencies via `pip install -r requirements.txt` in a fresh environment (e.g. Colab)."""


def ensure_pkg():
    """No-op placeholder; run pip manually or automate here if desired."""
    return None


ensure_pkg()


In [ ]:
"""Imports and global configuration for reproducibility.

Sets USE_TF=0 before HuggingFace loads, and forces trainer_utils.is_tf_available() to False so Trainer.setup never imports TensorFlow (broken TF wheels still report as installed).
"""
import os
os.environ["USE_TF"] = "0"
import re
import random
import zipfile
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.base import clone

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import transformers.trainer_utils as _hf_trainer_utils
_hf_trainer_utils.is_tf_available = lambda: False
from datasets import Dataset

from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ZIP_PATH = os.path.join(os.getcwd(), "jigsaw-unintended-bias-in-toxicity-classification.zip")
TRAIN_CSV_NAME = "train.csv"
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

BASELINE_DIR = os.path.join(ARTIFACT_DIR, "baseline_distilbert")
POISON_DIR = os.path.join(ARTIFACT_DIR, "poisoned_distilbert")
REWEIGHT_DIR = os.path.join(ARTIFACT_DIR, "reweighted_distilbert")
OVERSAMPLE_DIR = os.path.join(ARTIFACT_DIR, "oversample_distilbert")
ISOTONIC_PATH = os.path.join(ARTIFACT_DIR, "isotonic_calibrator.joblib")

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5

"""Optional: set True only for quick smoke tests (not for submission)."""
FAST_DEBUG = False
if FAST_DEBUG:
    NUM_EPOCHS = 1


## Load data from the zip (stratified 100k train / 20k eval)

We read only the columns needed to limit memory. **Label:** `y = (target >= 0.5).astype(int)`. `train_test_split` returns `(larger_slice, test_size_slice)`; the 20k holdout is the **second** return value.



In [ ]:
"""Load `train.csv` from the competition zip without extracting the full archive to disk."""


def read_train_from_zip(zip_path: str, csv_name: str) -> pd.DataFrame:
    """Read selected columns for memory efficiency and normalize names."""
    usecols = [
        "comment_text",
        "target",
        "black",
        "white",
        "muslim",
        "jewish",
        "homosexual_gay_or_lesbian",
    ]
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, usecols=usecols)
    df = df.rename(columns={"target": "toxic"})
    df["comment_text"] = df["comment_text"].fillna("").astype(str)
    for c in ["toxic", "black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df["y"] = (df["toxic"] >= 0.5).astype(int)
    return df


df_all = read_train_from_zip(ZIP_PATH, TRAIN_CSV_NAME)
if FAST_DEBUG:
    eval_n = 2000
    train_n = 5000
    cap = eval_n + train_n + 5000
    df_all, _ = train_test_split(
        df_all, train_size=min(cap, len(df_all)), stratify=df_all["y"], random_state=SEED
    )
else:
    eval_n = 20000
    train_n = 100000

y = df_all["y"].values
train_remaining, eval_df = train_test_split(
    df_all, test_size=eval_n, stratify=y, random_state=SEED
)
y_rem = train_remaining["y"].values
train_df, _discard = train_test_split(
    train_remaining, train_size=train_n, stratify=y_rem, random_state=SEED
)

print("train_df:", train_df.shape, "eval_df:", eval_df.shape)
print("toxic rate train:", train_df["y"].mean(), "eval:", eval_df["y"].mean())


train_df: (100000, 8) eval_df: (20000, 8)
toxic rate train: 0.07997 eval: 0.07995


# Part 1 — Baseline DistilBERT classifier

We fine-tune `distilbert-base-uncased` with HuggingFace `Trainer` (cross-entropy). Tokenization: `max_length=128`, `truncation=True`.



In [19]:
"""Fine-tune DistilBERT using HuggingFace Trainer (Part 1 core)."""
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(examples, tokenizer_obj, max_length: int):
    """Tokenize comment batches for DistilBERT."""
    return tokenizer_obj(
        examples["comment_text"],
        truncation=True,
        max_length=max_length,
    )


train_ds = Dataset.from_pandas(train_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
eval_ds = Dataset.from_pandas(eval_df[["comment_text", "y"]].rename(columns={"y": "labels"}))

train_ds = train_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)
eval_ds = eval_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


def compute_metrics(eval_pred):
    """Compute accuracy, macro-F1, binary toxic F1, and AUC when defined."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    pr = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    out = {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_binary": f1_score(labels, preds, pos_label=1),
    }
    try:
        out["auc"] = roc_auc_score(labels, pr)
    except ValueError:
        out["auc"] = float("nan")
    return out


args = TrainingArguments(
    output_dir=os.path.join(ARTIFACT_DIR, "trainer_baseline"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    load_best_model_at_end=False,
    report_to=[],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
trainer.save_model(BASELINE_DIR)
tokenizer.save_pretrained(BASELINE_DIR)
print(train_result)


/Users/hamad/miniconda3/envs/uni/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

/Users/hamad/miniconda3/envs/uni/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/hamad/miniconda3/envs/uni/lib/python3.10/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.Data

  0%|          | 0/18750 [00:00<?, ?it/s]

...

{'eval_loss': 0.14568226039409637, 'eval_accuracy': 0.9478, 'eval_f1_macro': 0.7781081595198295, 'eval_f1_binary': 0.5840637450199203, 'eval_auc': 0.948101564347235, 'eval_runtime': 93.9512, 'eval_samples_per_second': 212.876, 'eval_steps_per_second': 13.305, 'epoch': 1.0}
{'loss': 0.1165, 'grad_norm': 1.8172632455825806, 'learning_rate': 1.3173333333333333e-05, 'epoch': 1.02}
{'loss': 0.1318, 'grad_norm': 4.798831939697266, 'learning_rate': 1.2960000000000001e-05, 'epoch': 1.06}
{'loss': 0.1122, 'grad_norm': 6.106404781341553, 'learning_rate': 1.2746666666666668e-05, 'epoch': 1.09}
{'loss': 0.1105, 'grad_norm': 0.5825077295303345, 'learning_rate': 1.2533333333333336e-05, 'epoch': 1.12}
{'loss': 0.1116, 'grad_norm': 5.020034313201904, 'learning_rate': 1.232e-05, 'epoch': 1.15}
{'loss': 0.1226, 'grad_norm': 6.212255001068115, 'learning_rate': 1.2106666666666667e-05, 'epoch': 1.18}
{'loss': 0.1073, 'grad_norm': 3.852322816848755, 'learning_rate': 1.1893333333333335e-05, 'epoch': 1.22}
{'

  0%|          | 0/1250 [00:00<?, ?it/s]

{'eval_loss': 0.1601027548313141, 'eval_accuracy': 0.9469, 'eval_f1_macro': 0.80911628966355, 'eval_f1_binary': 0.6469414893617021, 'eval_auc': 0.9496524324224568, 'eval_runtime': 86.4354, 'eval_samples_per_second': 231.387, 'eval_steps_per_second': 14.462, 'epoch': 2.0}
{'loss': 0.082, 'grad_norm': 13.572542190551758, 'learning_rate': 6.560000000000001e-06, 'epoch': 2.02}
{'loss': 0.0645, 'grad_norm': 3.5587246417999268, 'learning_rate': 6.346666666666668e-06, 'epoch': 2.05}
{'loss': 0.0789, 'grad_norm': 0.031445957720279694, 'learning_rate': 6.133333333333334e-06, 'epoch': 2.08}
{'loss': 0.0811, 'grad_norm': 0.6073643565177917, 'learning_rate': 5.92e-06, 'epoch': 2.11}
{'loss': 0.0783, 'grad_norm': 0.25725147128105164, 'learning_rate': 5.706666666666667e-06, 'epoch': 2.14}
{'loss': 0.0665, 'grad_norm': 0.14686298370361328, 'learning_rate': 5.493333333333334e-06, 'epoch': 2.18}
{'loss': 0.1019, 'grad_norm': 0.5529361963272095, 'learning_rate': 5.28e-06, 'epoch': 2.21}
{'loss': 0.0879,

  0%|          | 0/1250 [00:00<?, ?it/s]

{'eval_loss': 0.21961188316345215, 'eval_accuracy': 0.94495, 'eval_f1_macro': 0.8091347634166886, 'eval_f1_binary': 0.6481303930968361, 'eval_auc': 0.9449902269294373, 'eval_runtime': 86.3997, 'eval_samples_per_second': 231.482, 'eval_steps_per_second': 14.468, 'epoch': 3.0}
{'train_runtime': 5064.3409, 'train_samples_per_second': 59.238, 'train_steps_per_second': 3.702, 'train_loss': 0.11889864995320638, 'epoch': 3.0}
TrainOutput(global_step=18750, training_loss=0.11889864995320638, metrics={'train_runtime': 5064.3409, 'train_samples_per_second': 59.238, 'train_steps_per_second': 3.702, 'train_loss': 0.11889864995320638, 'epoch': 3.0})


In [ ]:
"""Baseline evaluation: metrics, ROC/PR curves, confusion matrix, threshold sweep."""
model.eval()
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

def hf_predict_proba_toxic(texts: List[str], m, tok, device_obj, max_len: int) -> np.ndarray:
    """Return toxic-class probabilities for a list of strings."""
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device_obj)
    with torch.no_grad():
        logits = m(**enc).logits
        pr = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
    return pr


texts_eval = eval_df["comment_text"].tolist()
y_true = eval_df["y"].to_numpy()
scores = hf_predict_proba_toxic(texts_eval, model, tokenizer, device, MAX_LENGTH)
y_pred_05 = (scores >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_true, y_pred_05))
print("F1 macro:", f1_score(y_true, y_pred_05, average="macro"))
print("F1 toxic (binary):", f1_score(y_true, y_pred_05, pos_label=1))
print("AUC-ROC:", roc_auc_score(y_true, scores))

cm = confusion_matrix(y_true, y_pred_05)
print("Confusion matrix (thr=0.5):", cm)

fpr, tpr, thr_roc = roc_curve(y_true, scores)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label="ROC")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC — baseline")
plt.legend()
plt.show()

prec, rec, pr_thr = precision_recall_curve(y_true, scores)
plt.figure(figsize=(6, 5))
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall — baseline")
plt.show()

rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_hat = (scores >= t).astype(int)
    rows.append(
        {
            "threshold": t,
            "f1_macro": f1_score(y_true, y_hat, average="macro"),
            "f1_toxic": f1_score(y_true, y_hat, pos_label=1),
        }
    )
thr_tbl = pd.DataFrame(rows)
print(thr_tbl)

CHOSEN_THRESHOLD = 0.5
print("Chosen threshold for later parts:", CHOSEN_THRESHOLD)


### Part 1 — Threshold justification (model answer — align with your F1 sweep table)

**Choice:** **0.5** is a reasonable default: it matches the dataset’s toxicity binarization (≥0.5) and avoids picking an extreme without a clear policy reason. If your sweep shows **meaningfully higher macro-F1 or toxic-class F1** at e.g. **0.4** or **0.6**, you can adopt that threshold and cite the table.

**Trade-off:** **Lower τ** → more true positives (more abuse caught) but more **false positives** (innocent users limited or removed). **Higher τ** → fewer false positives but more **false negatives** (harmful content stays up). There is no universal optimum—only a **harm–error mix** the product and legal team can defend.

**Platform stance (example):** If the org prioritizes **user trust and false-accusation risk**, bias toward a **slightly higher** threshold or route borderline scores to **human review** (Part 5). If it prioritizes **minimizing visible abuse**, bias **lower** and accept more review load and false-positive complaints. **Edit:** Add **one sentence** naming your best τ from the printed sweep and which priority you chose.



# Part 2 — Bias audit (high-black vs reference cohort)

**Cohorts (evaluation subset):**
- **High-black:** `black >= 0.5`
- **Reference:** `black < 0.1` and `white >= 0.5`



In [ ]:
"""Bias audit metrics, disparate impact ratio, aif360 metrics, plots."""
cohort_high = eval_df["black"] >= 0.5
cohort_ref = (eval_df["black"] < 0.1) & (eval_df["white"] >= 0.5)

print("high-black cohort:", int(cohort_high.sum()))
print("reference cohort:", int(cohort_ref.sum()))


def cohort_metrics(y_true_arr: np.ndarray, y_score: np.ndarray, thr: float) -> Dict[str, float]:
    """TPR on toxic, FPR on non-toxic, FNR on toxic, precision on positives for a threshold."""
    y_hat = (y_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_arr, y_hat).ravel()
    tpr_toxic = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    fpr_nontox = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    fnr_toxic = fn / (tp + fn) if (tp + fn) > 0 else float("nan")
    prec_flag = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    return {
        "TPR_toxic": tpr_toxic,
        "FPR_nontoxic": fpr_nontox,
        "FNR_toxic": fnr_toxic,
        "Precision_flagged": prec_flag,
    }


def build_aif360_dataset(labels: np.ndarray, sens: np.ndarray) -> BinaryLabelDataset:
    """Build aif360 dataset; label=1 means toxic; race is the sensitive attribute."""
    df_b = pd.DataFrame({"label": labels.astype(float), "race": sens.astype(float)})
    return BinaryLabelDataset(
        favorable_label=0.0,
        unfavorable_label=1.0,
        df=df_b,
        label_names=["label"],
        protected_attribute_names=["race"],
        privileged_protected_attributes=[[0.0]],
    )


thr = CHOSEN_THRESHOLD
scores_eval = scores
y_hat = (scores_eval >= thr).astype(int)

m_high = cohort_metrics(y_true[cohort_high.values], scores_eval[cohort_high.values], thr)
m_ref = cohort_metrics(y_true[cohort_ref.values], scores_eval[cohort_ref.values], thr)

di = m_high["FPR_nontoxic"] / m_ref["FPR_nontoxic"] if m_ref["FPR_nontoxic"] > 0 else float("nan")

summary = pd.DataFrame(
    [
        {"cohort": "high_black", **m_high},
        {"cohort": "reference", **m_ref},
    ]
)
print(summary)
print("Disparate impact ratio (FPR_high / FPR_ref):", di)

sens = np.where(cohort_high, 1.0, 0.0)
sens[~cohort_high & ~cohort_ref] = 0.0

ds_true = build_aif360_dataset(y_true, sens)
ds_pred = build_aif360_dataset(y_hat, sens)

cm_obj = ClassificationMetric(
    ds_true,
    ds_pred,
    unprivileged_groups=[{"race": 1.0}],
    privileged_groups=[{"race": 0.0}],
)
print("aif360 statistical parity difference:", cm_obj.statistical_parity_difference())
print("aif360 equal opportunity difference:", cm_obj.equal_opportunity_difference())

labels_bar = ["TPR", "FPR", "FNR"]
x = np.arange(len(labels_bar))
w = 0.35
vals_high = [m_high["TPR_toxic"], m_high["FPR_nontoxic"], m_high["FNR_toxic"]]
vals_ref = [m_ref["TPR_toxic"], m_ref["FPR_nontoxic"], m_ref["FNR_toxic"]]

plt.figure(figsize=(8, 5))
plt.bar(x - w / 2, vals_high, width=w, label="high-black")
plt.bar(x + w / 2, vals_ref, width=w, label="reference")
plt.xticks(x, labels_bar)
plt.ylabel("rate")
plt.title("Group error rates (chosen threshold)")
plt.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))


def plot_cm(ax, yt, yp, title):
    """Plot a confusion matrix heatmap."""
    cm_local = confusion_matrix(yt, yp)
    sns.heatmap(cm_local, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)


plot_cm(axes[0], y_true[cohort_high.values], y_hat[cohort_high.values], "high-black cohort")
plot_cm(axes[1], y_true[cohort_ref.values], y_hat[cohort_ref.values], "reference cohort")
plt.tight_layout()
plt.show()


### Part 2 — Interpretation (model answer — tighten to your printed metrics)

**Largest disparity:** Often **FPR on non-toxic comments** differs most between cohorts (check **disparate impact ratio** and **SPD**). **EOD** shows whether **TPR** is skewed too, or if the main issue is **who gets falsely flagged**.

**Direction:** **Higher FPR** on the high-black cohort ⇒ **over-flagging** (innocent comments penalized). **Higher FNR** there ⇒ **under-flagging** (toxic content left up). **Both** can show up; your bar chart and confusion matrices show which dominates.

**Harm / risk:** Over-flagging chills speech and erodes trust where errors cluster; under-flagging leaves abuse targets less protected. For the platform, over-flagging raises **fairness and bias** exposure; under-flagging raises **safety and reputation** exposure.

**Edit for submission:** Replace “often” with your **biggest gap** (e.g. FPR ratio, SPD, or EOD) in **one sentence** tied to your outputs.



# Part 3 — Adversarial attacks (from scratch)

**Attack 1:** `perturb(text)` implements ZWS insertion, Cyrillic homoglyphs, and random character duplication.

**Attack 2:** flip **5%** of training labels and retrain **from `distilbert-base-uncased`**, not from the Part 1 fine-tuned checkpoint.



In [ ]:
"""Character-level evasion attack evaluation (Attack 1)."""
HOMOGLYPH = str.maketrans(
    {
        "a": "а",
        "e": "е",
        "o": "о",
        "c": "с",
        "p": "р",
        "x": "х",
    }
)

TOXICISH = re.compile(
    r"\b(?:kill|die|hate|stupid|idiot|ugly|loser|moron|rape|nazi|terror)\w*\b",
    re.IGNORECASE,
)


def perturb(text: str) -> str:
    """Insert ZWS into toxic-looking tokens, apply homoglyphs, duplicate ~20% of chars."""
    if text is None:
        return ""

    def zws_word(w: str) -> str:
        """Insert zero-width spaces every 2–3 characters."""
        out = []
        step = 2
        i = 0
        while i < len(w):
            chunk = w[i : i + step]
            out.append(chunk)
            if i + step < len(w):
                out.append("​")
            i += step
            step = 3 if step == 2 else 2
        return "".join(out)

    def process_segment(seg: str) -> str:
        """Homoglyph map plus random intra-word duplication."""
        seg = seg.translate(HOMOGLYPH)
        out_chars: List[str] = []
        for ch in seg:
            out_chars.append(ch)
            if ch.isalpha() and random.random() < 0.20:
                out_chars.append(ch)
        return "".join(out_chars)

    pieces: List[str] = []
    last = 0
    for m in TOXICISH.finditer(text):
        pieces.append(process_segment(text[last : m.start()]))
        pieces.append(process_segment(zws_word(m.group(0))))
        last = m.end()
    pieces.append(process_segment(text[last:]))
    return "".join(pieces)


attack_pool = eval_df.copy()
attack_pool["score"] = scores
attack_pool["y_hat"] = y_hat
cand = attack_pool[(attack_pool["y_hat"] == 1) & (attack_pool["score"] >= 0.7)]
if len(cand) > 0:
    cand = cand.sample(n=min(500, len(cand)), random_state=SEED)
else:
    cand = cand

pert_texts = [perturb(t) for t in cand["comment_text"].tolist()]
scores_before = cand["score"].to_numpy() if len(cand) else np.array([])
scores_after = (
    hf_predict_proba_toxic(pert_texts, model, tokenizer, device, MAX_LENGTH) if len(cand) else np.array([])
)
y_after = (scores_after >= thr).astype(int) if len(cand) else np.array([])

asr = float(np.mean(y_after == 0)) if len(cand) > 0 else float("nan")
print("Attack-1 candidates:", len(cand))
print("ASR (fraction no longer flagged at same threshold):", asr)
print("mean confidence before:", float(np.mean(scores_before)) if len(cand) else float("nan"))
print("mean confidence after:", float(np.mean(scores_after)) if len(cand) else float("nan"))


In [ ]:
"""Label-flip poisoning retrain (Attack 2)."""
poison_df = train_df.copy()
rng = np.random.RandomState(SEED)
flip_mask = rng.rand(len(poison_df)) < 0.05
poison_df.loc[flip_mask, "y"] = 1 - poison_df.loc[flip_mask, "y"]

poison_ds = Dataset.from_pandas(poison_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
poison_ds = poison_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)

model_poison = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_p = clone(args)
args_p.output_dir = os.path.join(ARTIFACT_DIR, "trainer_poison")

trainer_p = Trainer(
    model=model_poison,
    args=args_p,
    train_dataset=poison_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_p.train()
trainer_p.save_model(POISON_DIR)

model_poison.eval()
model_poison.to(device)
scores_poison_eval = hf_predict_proba_toxic(texts_eval, model_poison, tokenizer, device, MAX_LENGTH)
y_poison_hat = (scores_poison_eval >= thr).astype(int)


def fnr_toxic(yt, yp):
    """False negative rate restricted to truly toxic comments."""
    tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
    return fn / (fn + tp) if (fn + tp) > 0 else float("nan")


compare = pd.DataFrame(
    [
        {
            "model": "clean",
            "accuracy": accuracy_score(y_true, y_hat),
            "f1_macro": f1_score(y_true, y_hat, average="macro"),
            "fnr_toxic": fnr_toxic(y_true, y_hat),
        },
        {
            "model": "poisoned",
            "accuracy": accuracy_score(y_true, y_poison_hat),
            "f1_macro": f1_score(y_true, y_poison_hat, average="macro"),
            "fnr_toxic": fnr_toxic(y_true, y_poison_hat),
        },
    ]
)
print(compare)


### Part 3 — Operational risk (model answer — tie to your ASR / metric table)

**Evasion** needs no access to training—only the ability to post text. It is **cheap to automate** (scripts, templates), scales with **volume**, and is the **default threat** for any public moderation API or open comment box. Impact is **per-comment** but can still be severe at scale (spam, coordinated campaigns).

**Poisoning** needs **write access to training data or the fine-tuning pipeline** (compromised MLOps, malicious insider, poisoned vendor dataset). It is **rarer** and harder to pull off, but can **shift model behavior for everyone** (systemic FNs/FPs), not just one post.

**Which is “more dangerous”?** For **typical external abuse**, **evasion** is the more **realistic and frequent** operational problem—defenses should include **input normalization**, **robustness testing**, and **human review** on edge cases. **Poisoning** is the more **realistic high-severity insider/supply-chain** scenario—defenses should emphasize **data provenance, access control, auditing, and retraining checks**. **Edit:** One sentence comparing **your ASR** vs **your poisoned-model FNR/F1 drop** to say which risk worries you more for *this* deployment.



# Part 4 — Mitigation

Three interventions:
1. **Reweighing (aif360)** + `WeightedTrainer`
2. **ThresholdOptimizer (fairlearn, equalized_odds)** + Pareto-style sweep
3. **Oversampling** high-black training comments (`black >= 0.5`) with **3× duplicates**

For reweighing, rows outside the two defined cohorts are treated as **privileged** for weighting (documented engineering simplification).



In [ ]:
"""Reweighing instance weights and retrain DistilBERT with per-example loss scaling."""


def protected_for_fairness(df: pd.DataFrame) -> np.ndarray:
    """Sensitive attribute: 1 = high-black, 0 = reference, 0 = other (privileged default)."""
    hi = df["black"].to_numpy() >= 0.5
    ref = (df["black"].to_numpy() < 0.1) & (df["white"].to_numpy() >= 0.5)
    s = np.zeros(len(df), dtype=float)
    s[hi] = 1.0
    s[ref] = 0.0
    s[~(hi | ref)] = 0.0
    return s


sens_train = protected_for_fairness(train_df)
rw_df = pd.DataFrame({"label": train_df["y"].astype(float), "race": sens_train.astype(float)})
bld_train = BinaryLabelDataset(
    favorable_label=0.0,
    unfavorable_label=1.0,
    df=rw_df,
    label_names=["label"],
    protected_attribute_names=["race"],
    privileged_protected_attributes=[[0.0]],
)
rw = Reweighing(unprivileged_groups=[{"race": 1.0}], privileged_groups=[{"race": 0.0}])
rw_tr = rw.fit_transform(bld_train)
instance_weights = np.asarray(rw_tr.instance_weights).reshape(-1)


class WeightedTrainer(Trainer):
    """HF Trainer variant applying non-uniform instance weights to CE loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Weighted mean cross-entropy."""
        labels = inputs.pop("labels")
        weights = inputs.pop("weights", None)
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        loss = loss_fct(logits.view(-1, 2), labels.view(-1))
        if weights is not None:
            w = weights.view(-1).to(loss.device).float()
            loss = (loss * w).sum() / (w.sum() + 1e-8)
        return (loss.mean(), outputs) if return_outputs else loss.mean()


train_rw = Dataset.from_pandas(
    train_df[["comment_text", "y"]].rename(columns={"y": "labels"})
)
train_rw = train_rw.add_column("weights", instance_weights.tolist())
train_rw = train_rw.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)


class WeightedCollator:
    """Pad token batches while preserving per-example loss weights."""

    def __init__(self, tokenizer_obj):
        self.tokenizer = tokenizer_obj

    def __call__(self, features):
        """Return a batch dict including `weights` aligned with `labels`."""
        weights = torch.tensor([float(f["weights"]) for f in features], dtype=torch.float)
        fe = [{k: v for k, v in f.items() if k != "weights"} for f in features]
        batch = self.tokenizer.pad(fe, padding=True, return_tensors="pt")
        batch["weights"] = weights
        return batch


weighted_collator = WeightedCollator(tokenizer)

model_rw = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_rw = clone(args)
args_rw.output_dir = os.path.join(ARTIFACT_DIR, "trainer_reweight")
trainer_rw = WeightedTrainer(
    model=model_rw,
    args=args_rw,
    train_dataset=train_rw,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=weighted_collator,
    compute_metrics=compute_metrics,
)
trainer_rw.train()
trainer_rw.save_model(REWEIGHT_DIR)

model_rw.eval()
model_rw.to(device)
scores_rw = hf_predict_proba_toxic(texts_eval, model_rw, tokenizer, device, MAX_LENGTH)
y_rw_hat = (scores_rw >= thr).astype(int)


In [ ]:
"""fairlearn ThresholdOptimizer on baseline scores (equalized odds, prefit)."""
from sklearn.base import BaseEstimator, ClassifierMixin


class ScoresClassifier(BaseEstimator, ClassifierMixin):
    """Emit fixed probabilities from a precomputed score vector (length must match fit/predict data)."""

    def __init__(self, scores: np.ndarray):
        self.scores = scores.astype(float)
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        """No-op fit for prefit ThresholdOptimizer workflows."""
        return self

    def predict_proba(self, X):
        """Return (n,2) probabilities aligned to rows of X."""
        n = len(X)
        s = self.scores[:n]
        return np.column_stack([1.0 - s, s])


sens_eval = protected_for_fairness(eval_df)
mask_two = (sens_eval == 0.0) | (sens_eval == 1.0)
X_dummy = np.zeros((int(mask_two.sum()), 1))
y_sub = y_true[mask_two]
s_sub = sens_eval[mask_two]
scores_sub = scores[mask_two]

base_scores = ScoresClassifier(scores_sub)
post = ThresholdOptimizer(
    estimator=base_scores,
    constraints="equalized_odds",
    prefit=True,
    predict_method="predict_proba",
)
post.fit(X_dummy, y_sub, sensitive_features=s_sub)
y_post = post.predict(X_dummy)


In [ ]:
"""Oversample high-black training rows (3× duplicates) and retrain."""
hb_train = train_df[train_df["black"] >= 0.5]
extra = pd.concat([hb_train] * 3, ignore_index=True) if len(hb_train) else hb_train
train_os = pd.concat([train_df, extra], ignore_index=True)

train_os_ds = Dataset.from_pandas(train_os[["comment_text", "y"]].rename(columns={"y": "labels"}))
train_os_ds = train_os_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)

model_os = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_os = clone(args)
args_os.output_dir = os.path.join(ARTIFACT_DIR, "trainer_oversample")
trainer_os = Trainer(
    model=model_os,
    args=args_os,
    train_dataset=train_os_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_os.train()
trainer_os.save_model(OVERSAMPLE_DIR)

model_os.eval()
model_os.to(device)
scores_os = hf_predict_proba_toxic(texts_eval, model_os, tokenizer, device, MAX_LENGTH)
y_os_hat = (scores_os >= thr).astype(int)


In [ ]:
"""Pareto-style sweep: approximate fairness–F1 trade-offs via group threshold grids."""


def eo_diff(y_true_arr, y_hat_arr, sens_arr):
    """Equalized odds difference on rows with sensitive attribute in {0,1}."""
    m = (sens_arr == 0.0) | (sens_arr == 1.0)
    return equalized_odds_difference(
        y_true=y_true_arr[m],
        y_pred=y_hat_arr[m],
        sensitive_features=sens_arr[m],
    )


def spd_diff(y_true_arr, y_hat_arr, sens_arr):
    """Demographic parity difference on the same masked subset."""
    m = (sens_arr == 0.0) | (sens_arr == 1.0)
    return demographic_parity_difference(
        y_true=y_true_arr[m],
        y_pred=y_hat_arr[m],
        sensitive_features=sens_arr[m],
    )


def best_f1_under_eod(y_true_arr, scores_arr, sens_arr, tol: float) -> Tuple[float, float, np.ndarray]:
    """Greedy grid over group-specific thresholds subject to an EOD tolerance."""
    g0 = sens_arr == 0.0
    g1 = sens_arr == 1.0
    thr0_list = np.linspace(0.05, 0.95, 25)
    thr1_list = np.linspace(0.05, 0.95, 25)
    best_f1 = -1.0
    best_eod = float("nan")
    best_hat = None
    for t0 in thr0_list:
        for t1 in thr1_list:
            y_hat = np.zeros_like(y_true_arr)
            y_hat[g0] = (scores_arr[g0] >= t0).astype(int)
            y_hat[g1] = (scores_arr[g1] >= t1).astype(int)
            eod = abs(eo_diff(y_true_arr, y_hat, sens_arr))
            if eod <= tol + 1e-6:
                f1 = f1_score(y_true_arr, y_hat, average="macro")
                if f1 > best_f1:
                    best_f1 = f1
                    best_eod = eod
                    best_hat = y_hat
    if best_hat is None:
        y_hat = (scores_arr >= 0.5).astype(int)
        return f1_score(y_true_arr, y_hat, average="macro"), abs(eo_diff(y_true_arr, y_hat, sens_arr)), y_hat
    return best_f1, float(best_eod), best_hat


tols = np.linspace(0.0, 0.3, 11)
pareto_x = []
pareto_y = []
for tol in tols:
    f1, eod, _ = best_f1_under_eod(y_true, scores, sens_eval, float(tol))
    pareto_x.append(eod)
    pareto_y.append(f1)

plt.figure(figsize=(7, 5))
plt.scatter(pareto_x, pareto_y)
for xi, yi, ti in zip(pareto_x, pareto_y, tols):
    plt.annotate(f"{ti:.2f}", (xi, yi), textcoords="offset points", xytext=(4, 4), fontsize=8)
plt.xlabel("|equalized odds difference| (proxy via group thresholds)")
plt.ylabel("overall macro-F1")
plt.title("Pareto-style sweep (post-processing intuition)")
plt.show()


In [ ]:
"""Mitigation comparison table (baseline + three techniques)."""
sens_eval_full = protected_for_fairness(eval_df)


def cohort_metrics_labels(y_true_arr: np.ndarray, y_hat_arr: np.ndarray) -> Dict[str, float]:
    """Group metrics from hard predictions (matches post-processed thresholds)."""
    tn, fp, fn, tp = confusion_matrix(y_true_arr, y_hat_arr).ravel()
    tpr_toxic = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    fpr_nontox = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    fnr_toxic = fn / (tp + fn) if (tp + fn) > 0 else float("nan")
    prec_flag = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    return {
        "TPR_toxic": tpr_toxic,
        "FPR_nontoxic": fpr_nontox,
        "FNR_toxic": fnr_toxic,
        "Precision_flagged": prec_flag,
    }


def table_row(name: str, y_hat_arr: np.ndarray) -> Dict[str, float]:
    """One summary row for the Part 4 comparison table."""
    hi = eval_df["black"].to_numpy() >= 0.5
    ref = (eval_df["black"].to_numpy() < 0.1) & (eval_df["white"].to_numpy() >= 0.5)
    mhi = cohort_metrics_labels(y_true[hi], y_hat_arr[hi])
    mrf = cohort_metrics_labels(y_true[ref], y_hat_arr[ref])
    return {
        "model": name,
        "overall_f1_macro": f1_score(y_true, y_hat_arr, average="macro"),
        "highblack_FPR": mhi["FPR_nontoxic"],
        "reference_FPR": mrf["FPR_nontoxic"],
        "SPD": spd_diff(y_true, y_hat_arr, sens_eval_full),
        "EOD": eo_diff(y_true, y_hat_arr, sens_eval_full),
    }


rows_tbl = [
    table_row("baseline", y_hat),
    table_row("reweight", y_rw_hat),
    table_row("oversample", y_os_hat),
]

y_post_full = y_hat.copy()
y_post_full[mask_two] = y_post
rows_tbl.append(table_row("thresh_opt (subset)", y_post_full))

summary_tbl = pd.DataFrame(rows_tbl)
print(summary_tbl)

best_name = summary_tbl.sort_values("overall_f1_macro", ascending=False).iloc[0]["model"]
print("best overall F1 among rows:", best_name)

name_to_dir = {
    "baseline": BASELINE_DIR,
    "reweight": REWEIGHT_DIR,
    "oversample": OVERSAMPLE_DIR,
    "thresh_opt (subset)": REWEIGHT_DIR,
}
mitigated_rank = summary_tbl[summary_tbl["model"] != "baseline"].sort_values(
    "overall_f1_macro", ascending=False
)
best_mitigated_name = str(mitigated_rank.iloc[0]["model"]) if len(mitigated_rank) else "reweight"
BEST_MITIGATED_DIR = name_to_dir.get(best_mitigated_name, REWEIGHT_DIR)
print("Pipeline backbone (best mitigated row):", best_mitigated_name, "->", BEST_MITIGATED_DIR)


### Part 4 — Demographic parity vs equalized odds (model answer — plug in your `prev` table)

**Demographic parity (DP)** targets **equal predicted-positive rates** across cohorts (who gets flagged overall). **Equalized odds (EO)** targets **equal TPR and FPR** (same errors given the true label).

If **toxic prevalence** differs between cohorts (see your printed **P_toxic**), **DP and EO usually cannot both hold** for one fixed scoring rule: matching **overall flag rates** pushes you toward different **conditional** error rates than matching **TPR/FPR**, unless you add **group-specific thresholds** or accept **big accuracy loss**.

**Edit:** One sentence with your two **P_toxic** values and whether **SPD** vs **EOD** trade off after Technique 2 in your run.



In [ ]:
"""Empirical toxic prevalence by cohort on the evaluation subset."""
prev = pd.DataFrame(
    {
        "cohort": ["high_black", "reference"],
        "P_toxic": [
            float(y_true[cohort_high.values].mean()) if cohort_high.any() else float("nan"),
            float(y_true[cohort_ref.values].mean()) if cohort_ref.any() else float("nan"),
        ],
    }
)
print(prev)


# Part 5 — Guardrail pipeline (`pipeline.py`)

`ModerationPipeline.predict` applies the regex blocklist, then **isotonic-calibrated** transformer probabilities, then routes **0.4–0.6** to human review.

The notebook fits `CalibratedClassifierCV(..., method='isotonic')` for reporting, then saves a standalone `IsotonicRegression` mapping for `pipeline.py` inference (avoids pickling duplicate transformer weights inside sklearn).



In [ ]:
"""Demonstrate `ModerationPipeline` on 1000 evaluation comments + layer distribution plots."""
from pipeline import ModerationPipeline, input_filter, HFProbabilityEstimator

mitigated_tokenizer = AutoTokenizer.from_pretrained(BEST_MITIGATED_DIR)
mitigated_model = AutoModelForSequenceClassification.from_pretrained(BEST_MITIGATED_DIR)
mitigated_model.eval()
mitigated_model.to(device)

calib_eval_idx = eval_df.sample(n=min(3000, len(eval_df)), random_state=SEED).index
calib_texts = eval_df.loc[calib_eval_idx, "comment_text"].tolist()
calib_y = eval_df.loc[calib_eval_idx, "y"].to_numpy()
raw_cal_scores = hf_predict_proba_toxic(calib_texts, mitigated_model, mitigated_tokenizer, device, MAX_LENGTH)

base_est = HFProbabilityEstimator(mitigated_model, mitigated_tokenizer, device, MAX_LENGTH)
cc = CalibratedClassifierCV(base_est, method="isotonic", cv="prefit")
cc.fit(calib_texts, calib_y)

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(raw_cal_scores, calib_y)
joblib.dump(iso, ISOTONIC_PATH)

PIPE_MODEL_DIR = os.path.join(ARTIFACT_DIR, "best_mitigated_for_pipeline")
mitigated_model.save_pretrained(PIPE_MODEL_DIR)
mitigated_tokenizer.save_pretrained(PIPE_MODEL_DIR)

pipe = ModerationPipeline(model_dir=PIPE_MODEL_DIR, calibrator_path=ISOTONIC_PATH, device=str(device))

demo_n = min(1000, len(eval_df))
demo_df = eval_df.sample(n=demo_n, random_state=SEED)

counts = {"input_filter": 0, "model_block": 0, "model_allow": 0, "model_review": 0}
cat_counts = {
    "direct_threat": 0,
    "self_harm_directed": 0,
    "doxxing_stalking": 0,
    "dehumanization": 0,
    "coordinated_harassment": 0,
}

for t in demo_df["comment_text"].tolist():
    r = input_filter(t)
    if r is not None:
        counts["input_filter"] += 1
        cat = r.get("category")
        if cat in cat_counts:
            cat_counts[cat] += 1
        continue
    d = pipe.predict(t)
    if d["decision"] == "block":
        counts["model_block"] += 1
    elif d["decision"] == "allow":
        counts["model_allow"] += 1
    else:
        counts["model_review"] += 1

print("Layer counts:", counts)
print("Regex category counts:", cat_counts)

plt.figure(figsize=(6, 6))
plt.pie(list(counts.values()), labels=list(counts.keys()), autopct="%1.1f%%")
plt.title("Fraction of decisions by layer (demo)")
plt.show()

auto_texts = []
auto_true = []
review_true = []
for txt, yt in zip(demo_df["comment_text"].tolist(), demo_df["y"].tolist()):
    if input_filter(txt) is not None:
        continue
    d = pipe.predict(txt)
    if d["decision"] in ("block", "allow"):
        auto_texts.append(txt)
        auto_true.append(yt)
    else:
        review_true.append(yt)

if auto_texts:
    y_auto_true = np.array(auto_true)
    y_auto_hat = []
    p_obj = ModerationPipeline(
        model_dir=PIPE_MODEL_DIR, calibrator_path=ISOTONIC_PATH, device=str(device)
    )
    for txt in auto_texts:
        pr = p_obj._calibrated_toxic_prob(txt)
        y_auto_hat.append(1 if pr >= 0.6 else 0)
    y_auto_hat = np.array(y_auto_hat)
    print(
        "Auto-actioned F1/precision/recall:",
        f1_score(y_auto_true, y_auto_hat, pos_label=1),
        precision_score(y_auto_true, y_auto_hat, pos_label=1, zero_division=0),
        recall_score(y_auto_true, y_auto_hat, pos_label=1, zero_division=0),
    )

if review_true:
    print("Review queue toxic rate (ground truth):", float(np.mean(review_true)))


In [ ]:
"""Uncertainty band sensitivity analysis (required)."""


def pipeline_counts(block_hi: float, allow_lo: float):
    """Count review vs auto model decisions excluding regex blocks."""
    p = ModerationPipeline(
        model_dir=PIPE_MODEL_DIR,
        calibrator_path=ISOTONIC_PATH,
        device=str(device),
        block_hi=block_hi,
        allow_lo=allow_lo,
    )
    rev = 0
    auto = 0
    for txt in demo_df["comment_text"].tolist():
        if input_filter(txt) is not None:
            continue
        d = p.predict(txt)
        if d["decision"] == "review":
            rev += 1
        elif d["decision"] in ("block", "allow"):
            auto += 1
    return rev, auto


def auto_f1_at(block_hi: float, allow_lo: float) -> float:
    """F1 on toxic class for auto block/allow using block threshold on calibrated scores."""
    p = ModerationPipeline(
        model_dir=PIPE_MODEL_DIR,
        calibrator_path=ISOTONIC_PATH,
        device=str(device),
        block_hi=block_hi,
        allow_lo=allow_lo,
    )
    ys = []
    yh = []
    for txt, yt in zip(demo_df["comment_text"].tolist(), demo_df["y"].tolist()):
        if input_filter(txt) is not None:
            continue
        d = p.predict(txt)
        if d["decision"] not in ("block", "allow"):
            continue
        pr = p._calibrated_toxic_prob(txt)
        ys.append(yt)
        yh.append(1 if pr >= block_hi else 0)
    if not ys:
        return float("nan")
    return f1_score(np.array(ys), np.array(yh), pos_label=1)


configs = [
    ("default 0.4–0.6", 0.6, 0.4),
    ("narrow 0.45–0.55", 0.55, 0.45),
    ("wide 0.3–0.7", 0.7, 0.3),
]

rows_sens = []
for name, hi, lo in configs:
    rev, auto = pipeline_counts(hi, lo)
    rows_sens.append(
        {
            "band": name,
            "review_count": rev,
            "auto_count": auto,
            "auto_f1_toxic": auto_f1_at(hi, lo),
        }
    )

print(pd.DataFrame(rows_sens))


### Part 5 — Uncertainty band choice (model answer — cite your sensitivity table)

The **0.4–0.6** band routes **only middling calibrated probabilities** to humans; **≤0.4** allow and **≥0.6** block stay automatic. That is a **middle ground**: wide enough to catch model doubt after **isotonic calibration**, not so wide that almost everything hits review.

**Narrow (e.g. 0.45–0.55):** fewer **review_count** rows, but more borderline cases are **auto-blocked or auto-allowed**—risky if calibration is imperfect. **Wide (e.g. 0.3–0.7):** **review_count** grows (cost, latency), but fewer decisions are forced when the model is unsure; **auto_f1_toxic** may shift because the auto subset changes.

**Default choice:** Keep **0.4–0.6** unless your table shows **auto_f1** collapsing at default while narrow improves it **and** ops can absorb mis-automation risk—or unless review volume is untenable and you **narrow** with extra safeguards (sampling, appeals). **Edit:** One sentence comparing **review_count** and **auto_f1_toxic** across the three rows you printed.

